In [1]:
import sys
import os
import json
from pathlib import Path

# 1. Path Setup
_PROJECT_ROOT = Path.cwd().parents[2]
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

_RQ1_DIR = Path.cwd().parent
if str(_RQ1_DIR) not in sys.path:
    sys.path.insert(0, str(_RQ1_DIR))

from evaluate_rq1 import *
print(f"✓ Success! Project root set to: {_PROJECT_ROOT}")

# 2. Define File Paths
# Point this directly to your test file
TEST_DATA_PATH = os.path.join(_PROJECT_ROOT, "datasets", "wikidata_testset_RQ1.json") 
CHECKPOINT_PATH = os.path.join(_PROJECT_ROOT, "checkpoints", "wikidata", "best_model.pt")

TOPKS = [3, 5, 10, 20]
BATCH = 32

# 3. Load the test dataset directly
test_data = load_triplets(TEST_DATA_PATH)

print(f"Dataset Loaded -> Test Split: {len(test_data)}")

# 4. Build corpus and eval items using ONLY the test data
global_corpus, seen_q = build_global_corpus(test_data)
eval_items = build_eval_items(test_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

print(f"Evaluation Ready -> Anchors: {len(eval_items)} | Corpus: {len(global_corpus)}")
# Enable per-bin lexical stratification for every model below:
#   [0,0.3) adversarial | [0.3,0.5) divergent | [0.5,0.8) moderate | [0.8,1.0) near-identical
import evaluate_rq1
evaluate_rq1.STRATIFY_BINS = [0.3, 0.5, 0.8]  # cumulative: cos < threshold


/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /local/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /local/upb/users/z/zahera/profiles/unix/cs/.conda/envs/s

✓ Success! Project root set to: /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings
Dataset Loaded -> Test Split: 1063
Evaluation Ready -> Anchors: 1063 | Corpus: 3837


In [2]:
# SPARQLCE
evaluate_sparqlce_checkpoint(CHECKPOINT_PATH, eval_items, global_corpus, relevance_scores_per_item, TOPKS, batch_size=BATCH)

# TF-IDF
evaluate_tfidf(eval_items, global_corpus, relevance_scores_per_item, TOPKS)

# BM25
evaluate_bm25(eval_items, global_corpus, relevance_scores_per_item, TOPKS)

# E5
evaluate_sentence_transformer('intfloat/e5-base-v2', 'E5-base-v2', eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=lambda q: f'query: {q}')

# Qwen2.5-Coder (generative code LLM used as an embedder, last-token pooling;
# 0.5B ≈ scale-matched to the dense 0.6B Qwen3-Embedding it replaces)
evaluate_code_llm('Qwen/Qwen2.5-Coder-0.5B', 'Qwen2.5-Coder-0.5B', eval_items, global_corpus, relevance_scores_per_item, TOPKS)

Loading model from /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings/checkpoints/wikidata/best_model.pt...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 929.26it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



SPARQLCE
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.2656  0.3794  0.2643  0.4575  0.5005    0.2656  0.4337
@5     0.2060  0.4287  0.2278  0.4696  0.5532    0.2060  0.4477
@10    0.1448  0.4847  0.1786  0.4797  0.6275    0.1448  0.4633
@20    0.1035  0.5563  0.1431  0.4848  0.6999    0.1035  0.4807
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.2174      0.2572      471   0.1079   0.1249   0.1379   0.1463
            <0.5    0.2681      0.2718      616   0.1347   0.1498   0.1639   0.1718
            <0.8    0.3182      0.2779      788   0.2544   0.2692   0.2814   0.2887

Baseline: TF-IDF
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
------------------------

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 866.73it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



E5-base-v2
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.2038  0.2883  0.2001  0.3663  0.4083    0.2038  0.3346
@5     0.1522  0.3154  0.1671  0.3745  0.4440    0.1522  0.3375
@10    0.1038  0.3605  0.1299  0.3809  0.4920    0.1038  0.3468
@20    0.0697  0.4036  0.0985  0.3843  0.5409    0.0697  0.3550
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.2174      0.2572      471   0.0159   0.0191   0.0219   0.0252
            <0.5    0.2681      0.2718      616   0.0557   0.0631   0.0680   0.0726
            <0.8    0.3182      0.2779      788   0.1755   0.1837   0.1910   0.1954


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 928.88it/s, Materializing param=norm.weight]                              



Qwen2.5-Coder-0.5B
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1571  0.1855  0.1366  0.2777  0.3208    0.1571  0.2347
@5     0.1270  0.2227  0.1274  0.2863  0.3612    0.1270  0.2414
@10    0.0913  0.2719  0.1072  0.2955  0.4280    0.0913  0.2510
@20    0.0637  0.3266  0.0872  0.2995  0.4864    0.0637  0.2611
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.2174      0.2572      471   0.0280   0.0318   0.0370   0.0402
            <0.5    0.2681      0.2718      616   0.0817   0.0866   0.0943   0.0987
            <0.8    0.3182      0.2779      788   0.1559   0.1629   0.1707   0.1756


In [3]:
# Prior-literature SPARQL / code encoder baselines (zero-shot, mean-pooled):
#   CodeBERT (SparqlCE's own backbone), GraphCodeBERT (CodeBERT + data-flow
#   graph), SPBERT (BERT pretrained on SPARQL). Defined in evaluate_rq1.py as
#   SPARQL_ENCODER_BASELINES. The "mean pooling" notice each emits is correct
#   here (raw MLM encoders), not the LoRA-adapter bug.
evaluate_sparql_encoder_baselines(eval_items, global_corpus, relevance_scores_per_item, TOPKS, batch_size=BATCH)

No sentence-transformers model found with name microsoft/codebert-base. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1018.81it/s, Materializing param=pooler.dense.weight]                               



CodeBERT (zero-shot)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1562  0.2152  0.1500  0.2860  0.3198    0.1562  0.2522
@5     0.1214  0.2493  0.1321  0.2947  0.3584    0.1214  0.2598
@10    0.0836  0.2927  0.1044  0.3011  0.4083    0.0836  0.2689
@20    0.0580  0.3440  0.0824  0.3053  0.4694    0.0580  0.2799
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.2174      0.2572      471   0.0131   0.0136   0.0153   0.0175
            <0.5    0.2681      0.2718      616   0.0506   0.0555   0.0588   0.0628
            <0.8    0.3182      0.2779      788   0.1206   0.1277   0.1327   0.1367


No sentence-transformers model found with name microsoft/graphcodebert-base. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1455.87it/s, Materializing param=encoder.layer.11.output.dense.weight]              
RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect ide


GraphCodeBERT (zero-shot)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1803  0.2565  0.1768  0.3260  0.3706    0.1803  0.2942
@5     0.1362  0.2871  0.1505  0.3335  0.4036    0.1362  0.2998
@10    0.0932  0.3317  0.1173  0.3401  0.4516    0.0932  0.3099
@20    0.0633  0.3718  0.0897  0.3438  0.5052    0.0633  0.3182
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.2174      0.2572      471   0.0180   0.0185   0.0202   0.0226
            <0.5    0.2681      0.2718      616   0.0498   0.0538   0.0579   0.0620
            <0.8    0.3182      0.2779      788   0.1419   0.1483   0.1531   0.1576


No sentence-transformers model found with name razent/spbert-mlm-wso-base. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1500.28it/s, Materializing param=encoder.layer.11.output.dense.weight]              
BertModel LOAD REPORT from: razent/spbert-mlm-wso-base
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                       


SPBERT (zero-shot)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1907  0.2722  0.1884  0.3451  0.3819    0.1907  0.3118
@5     0.1447  0.2989  0.1585  0.3537  0.4186    0.1447  0.3164
@10    0.0982  0.3395  0.1228  0.3604  0.4666    0.0982  0.3243
@20    0.0646  0.3865  0.0920  0.3637  0.5146    0.0646  0.3325
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.2174      0.2572      471   0.0078   0.0097   0.0115   0.0140
            <0.5    0.2681      0.2718      616   0.0482   0.0534   0.0582   0.0624
            <0.8    0.3182      0.2779      788   0.1436   0.1521   0.1580   0.1624


In [4]:
display_name = "SBERT/MiniLM" 
model_id = "sentence-transformers/all-MiniLM-L6-v2"
evaluate_sentence_transformer(model_id, display_name, eval_items, global_corpus, relevance_scores_per_item, TOPKS)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 807.93it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



SBERT/MiniLM
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1734  0.2348  0.1646  0.3089  0.3481    0.1734  0.2769
@5     0.1334  0.2601  0.1415  0.3161  0.3810    0.1334  0.2807
@10    0.0940  0.3039  0.1144  0.3231  0.4346    0.0940  0.2900
@20    0.0659  0.3638  0.0919  0.3275  0.4995    0.0659  0.3030
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.2174      0.2572      471   0.0163   0.0186   0.0217   0.0239
            <0.5    0.2681      0.2718      616   0.0522   0.0578   0.0640   0.0682
            <0.8    0.3182      0.2779      788   0.1544   0.1619   0.1692   0.1736


In [5]:
display_name = "SBERT/MPNet"
model_id = "sentence-transformers/all-mpnet-base-v2"
evaluate_sentence_transformer(model_id, display_name, eval_items, global_corpus, relevance_scores_per_item, TOPKS)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 903.84it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



SBERT/MPNet
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1649  0.2206  0.1556  0.2943  0.3349    0.1649  0.2616
@5     0.1272  0.2505  0.1354  0.3013  0.3659    0.1272  0.2664
@10    0.0926  0.3061  0.1131  0.3096  0.4280    0.0926  0.2806
@20    0.0641  0.3574  0.0896  0.3136  0.4854    0.0641  0.2914
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.2174      0.2572      471   0.0163   0.0184   0.0211   0.0230
            <0.5    0.2681      0.2718      616   0.0555   0.0590   0.0630   0.0680
            <0.8    0.3182      0.2779      788   0.1394   0.1458   0.1526   0.1572


In [6]:
# E5-base-v2 (fine-tuned on SQuARE-Wikidata) — capacity-matched trained baseline
# Checkpoint saved by 05_e5_contrastive_finetune_square_wikidata.ipynb
E5_FT_PATH = os.path.join(_PROJECT_ROOT, "checkpoints", "e5_square_wikidata", "best_model")
evaluate_sentence_transformer(
    E5_FT_PATH,
    "E5-base-v2 (fine-tuned)",
    eval_items,
    global_corpus,
    relevance_scores_per_item,
    TOPKS,
    transform=lambda q: f"query: {q}",
)

No sentence-transformers model found with name /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings/checkpoints/e5_square_wikidata/best_model. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1069.13it/s, Materializing param=pooler.dense.weight]                              



E5-base-v2 (fine-tuned)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.2728  0.3809  0.2661  0.4735  0.4986    0.2728  0.4490
@5     0.2083  0.4122  0.2234  0.4806  0.5296    0.2083  0.4544
@10    0.1429  0.4521  0.1718  0.4853  0.5663    0.1429  0.4608
@20    0.0980  0.5028  0.1334  0.4884  0.6105    0.0980  0.4687
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.2174      0.2572      471   0.0350   0.0414   0.0469   0.0499
            <0.5    0.2681      0.2718      616   0.0974   0.1073   0.1161   0.1208
            <0.8    0.3182      0.2779      788   0.2580   0.2677   0.2735   0.2780
